# NeuroSafe - exploratory notebook

> Educational use only. Reports are not proof of causality and counts are not incidence rates.

Run `scripts/download_data.py` and `scripts/preprocess_data.py` first. If `data/raw/download_metadata.json` says `is_sample: true`, everything below is SYNTHETIC test data.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import plotly.express as px
from src import config, data_loader, preprocessing
from src.features import explode_drugs, explode_reactions

print(json.dumps({k: v for k, v in data_loader.data_source_info().items() if k != 'windows'}, indent=2)[:900])
df = data_loader.load_reports()
assert df is not None, 'Run the scripts in scripts/ first'
df.shape

## 1. Shape, types and missing values

In [ ]:
display(df.dtypes.to_frame('dtype'))
preprocessing.missing_value_summary(df)

## 2. Which study drugs appear in the sample? (capped sample - not real volumes)

In [ ]:
pairs = explode_drugs(df)
counts = pairs['drug'].value_counts().rename_axis('drug').reset_index(name='reports')
px.bar(counts, x='drug', y='reports', title='Report mentions per study drug (sample)')

## 3. Reports over time

In [ ]:
q = df.groupby('report_quarter').size().reset_index(name='reports')
px.line(q, x='report_quarter', y='reports', markers=True, title='Reports per quarter (latest quarters may be incomplete)')

## 4. Seriousness flag and top reported reactions

In [ ]:
print(df['serious_label'].value_counts(normalize=True).round(3))
top = explode_reactions(df)['reaction'].value_counts().head(15).rename_axis('reaction').reset_index(name='reports')
px.bar(top.iloc[::-1], x='reports', y='reaction', orientation='h', title='Top 15 reported reaction terms')

## 5. Demographics (valid values only)

In [ ]:
print(df['sex'].value_counts())
px.histogram(df.dropna(subset=['age_years']), x='age_years', nbins=30, title='Reported age (years)')

## 6. Notes to write in your own words
- What did you observe about reporting patterns?
- Which findings could be explained by reporting bias or the download cap rather than real differences?
- What would you need (denominators, validated cases) to say anything about risk?